# Kjemiske bibliotekerPython har titusenvis av ferdige bibliotek, og en god del av dem er skrevet spesielt for kjemi. Spørsmålet er ikke om de finnes, men hvilke av dem som er verdt tida di.Et bibliotek er verdt å lære seg når det gir deg noe du ikke kan lage selv på fem minutter. Det er i hovedsak tre ting som kvalifiserer:1. **Data.** Kuraterte, kildebelagte tall som noen har samla inn og kvalitetssikra. Du kan ikke finne på ioniseringsenergien til wolfram.2. **Maskineri.** Algoritmer og grafikk som er store nok til at det ville tatt uker å skrive dem selv.3. **Standardisering.** Formater og konvensjoner som resten av fagfeltet allerede bruker, slik at koden din kan snakke med andres.Bibliotek som *bare* pakker inn en beregning du burde kunne gjøre selv, kvalifiserer ikke. Da bytter du bort forståelse mot et API du sannsynligvis aldri får bruk for igjen.Dette skillet er viktigere nå enn det var før. En språkmodell skriver limkoden for deg på sekunder, så det å pugge syntaksen til et bibliotek er blitt nokså verdiløst. Det som har blitt *mer* verdt, er å vite hva som finnes, å kunne lese dokumentasjon, og å kunne avgjøre om det som kommer ut er fornuftig. Det er dét dette kapitlet handler om.Vi organiserer derfor kapitlet etter hva slags hjelp biblioteket gir, ikke etter navn:| Del | Hva biblioteket gir deg | Eksempler ||---|---|---|| 1 | Data | mendeleev, PubChemPy || 2 | Struktur og representasjon | RDKit || 3 | Ingenting du ikke kan skrive selv | (her skriver vi koden selv) || 4 | Numerikk du bør sette ut | pHcalc |

```{admonition} Læringsutbytte:class: noteEtter å ha arbeidet med denne delen av emnet, skal du kunne:1. Hente og utforske grunnstoffdata med `mendeleev`, både ett grunnstoff om gangen og som datasett.2. Hente data om kjemiske forbindelser fra PubChem med `pubchempy`.3. Representere molekyler fra SMILES og beregne molekylegenskaper med `RDKit`.4. Balansere en reaksjonslikning ved å løse et lineært likningssystem.5. Beregne pH i sammensatte løsninger med `pHcalc`, og kontrollere svaret mot din egen numeriske metode.6. Vurdere om et bibliotek er verdt å bruke, og når du heller bør skrive koden selv.```

## InstallasjonKjør denne cella én gang. Utropstegnet gjør at linja kjøres som en kommando i terminalen, ikke som Python-kode.

In [ ]:
!pip install mendeleev pubchempy rdkit pHcalc sympy

## Del 1: Bibliotek som gir deg data### MendeleevHvorfor bruker vi et bibliotek her? Fordi periodesystemet er et **datasett**, ikke en formel.Elektronegativiteten til svovel er ikke noe du kan regne deg fram til fra grunnprinsipper på en ettermiddag. Den er et resultat av målinger og modellvalg, og noen har gjort jobben med å samle inn verdiene, sjekke dem mot originalkildene og legge dem i en database. `mendeleev` gir deg tilgang til den jobben. Det er nøyaktig den typen hjelp du ikke kan skaffe deg på egen hånd, og derfor er biblioteket verdt tida.Grunnenheten er klassen `element`. En klasse er en oppskrift for å lage objekter. Hvert grunnstoffobjekt er laget etter samme oppskrift og har derfor de samme egenskapene, men med ulike verdier.

In [ ]:
from mendeleev import elementsvovel = element("S")          # Eller: element(16)print(svovel.name)print(svovel.symbol)print(svovel.atomic_number)print(svovel.atomic_weight)print(svovel.block, svovel.period, svovel.group_id)

Elektronegativitet er et *metodekall* og ikke en vanlig egenskap. Det er fordi elektronegativitet ikke er én størrelse, men flere ulike skalaer som bygger på ulike definisjoner. Du må derfor si hvilken du vil ha.

In [ ]:
print("Pauling: ", svovel.electronegativity("pauling"))print("Allen:   ", svovel.electronegativity("allen"))print("Mulliken:", svovel.electronegativity("mulliken"))

```{admonition} Underveisoppgave: Hvorfor er tallene så forskjellige?:class: tipDe tre tallene ovenfor ser ut som om skalaene er voldsomt uenige om hvor elektronegativt svovel er. Det er de ikke.Slå opp definisjonene av de tre skalaene. Hva er enheten til hver av dem? Hvorfor gir det ikke mening å sammenlikne tallene direkte, og hva *må* du gjøre før en slik sammenlikning blir meningsfull?```````{admonition} Løsningsforslag:class: dropdownPaulings skala er *enhetsløs*. Den er definert ut fra bindingsenergier og er skalert slik at fluor får verdien 3.98.Allens og Mullikens skalaer er derimot definert ut fra energier, og oppgis i **elektronvolt (eV)**. Allen bruker gjennomsnittlig valenselektronenergi, Mulliken bruker gjennomsnittet av ioniseringsenergi og elektronaffinitet.Tallene kan altså ikke sammenliknes direkte, like lite som du kan sammenlikne 20 grader Celsius med 293 kelvin ved å se på tallene alene. Det som *er* sammenliknbart, er **rekkefølgen** grunnstoffene får, og de relative avstandene mellom dem etter at skalaene er normert til samme intervall.Dette er et generelt poeng: en tallverdi fra et bibliotek betyr ingenting før du vet hvilken definisjon og hvilken enhet den hører til.````

#### Hele periodesystemet på én gangHvis du vil se på trender, er det tungvint å lage ett objekt per grunnstoff. Hvert kall på `element()` er et oppslag i databasen, så en løkke over alle grunnstoffene gjør 118 separate oppslag.`fetch_table` henter i stedet hele tabellen i én operasjon og gir deg den som en **pandas-dataframe**. Da kan du bruke alt du har lært om datahåndtering.

In [ ]:
from mendeleev.fetch import fetch_tableimport pandas as pdgrunnstoffer = fetch_table("elements")kolonner = ["atomic_number", "symbol", "name", "period", "group_id",            "block", "atomic_weight", "en_pauling", "covalent_radius_pyykko"]grunnstoffer[kolonner].head(12)

#### Hull i datasettetEkte datasett har hull. Det gjelder også kuraterte kjemidatabaser, og det er ikke slurv: noen verdier er rett og slett ikke definert eller ikke målt.

In [ ]:
mangler = grunnstoffer[grunnstoffer["en_pauling"].isna()]print("Antall grunnstoff uten Pauling-elektronegativitet:", len(mangler))mangler[["atomic_number", "symbol", "name", "group_id"]]

```{admonition} Underveisoppgave: Hvem mangler, og hvorfor?:class: tipSe på lista ovenfor.1. Én hel gruppe i periodesystemet er nesten fullstendig representert blant de manglende verdiene. Hvilken, og hva er den kjemiske grunnen til det?2. Resten av de manglende verdiene har en helt annen forklaring. Hva har disse grunnstoffene til felles?3. Lag et plott av Pauling-elektronegativitet mot atomnummer for hele periodesystemet. Hva skjer med punktene som mangler verdi? Blir det en feilmelding, eller forsvinner de stille?```````{admonition} Løsningsforslag:class: dropdown1. **Edelgassene.** Paulings skala er definert ut fra bindingsenergier, og de letteste edelgassene danner i praksis ikke bindinger som kan brukes til å definere en verdi. (Merk at de tyngste edelgassene, som xenon, faktisk har fått verdier, fordi de danner forbindelser med fluor og oksygen.)2. **De tyngste, kunstig framstilte grunnstoffene.** De finnes bare i mengder på noen få atomer om gangen og med halveringstider på sekunder eller mindre. Det er ingen som har rukket å måle bindingsenergier på dem.3. Punktene forsvinner **stille**. `matplotlib` gjør `None` om til `NaN`, og `NaN`-punkter tegnes ikke. Du får ingen feilmelding, og det er nettopp derfor du må se på dataene før du plotter dem.```pythonimport matplotlib.pyplot as pltplt.figure(figsize=(9, 4))plt.scatter(grunnstoffer["atomic_number"], grunnstoffer["en_pauling"], s=18)plt.xlabel("Atomnummer")plt.ylabel("Elektronegativitet (Pauling)")plt.title("Elektronegativitet i periodesystemet")plt.grid(alpha=0.3)plt.show()```````

#### Trender innafor en periodeMed dataene i en dataframe blir det enkelt å plukke ut akkurat den delen av periodesystemet du vil studere.

In [ ]:
import matplotlib.pyplot as pltperiode2 = grunnstoffer[grunnstoffer["period"] == 2]plt.figure(figsize=(7, 4))plt.plot(periode2["atomic_number"], periode2["en_pauling"],         marker="o", color="crimson")for _, rad in periode2.iterrows():    plt.annotate(rad["symbol"],                 (rad["atomic_number"], rad["en_pauling"]),                 textcoords="offset points", xytext=(0, 8), ha="center")plt.xlabel("Atomnummer")plt.ylabel("Elektronegativitet (Pauling)")plt.title("Elektronegativitet i andre periode")plt.grid(alpha=0.3)plt.show()

```{admonition} Underveisoppgave: Forklar to trender:class: tip1. Plottet ovenfor viser andre periode (Z = 3 til 10). Beskriv trenden og forklar den ut fra kjerneladning og skjerming. Hvorfor mangler neon et punkt?2. Lag tilsvarende plott for **gruppe 17** (halogenene) og for **gruppe 1**. Går trenden samme vei? Forklar.3. Lag ett plott der du viser periode 2 og periode 3 i samme koordinatsystem, med hver sin farge og merkelapp. Hva sier sammenlikninga om effekten av å legge til et helt elektronskall?```````{admonition} Løsningsforslag:class: dropdown1. Elektronegativiteten øker fra litium til fluor. Kjerneladninga øker med ett proton per steg, mens de nye elektronene går inn i det *samme* skallet og skjermer hverandre dårlig. Effektiv kjerneladning øker derfor, atomradien minker, og valenselektronene holdes hardere. Neon mangler verdi av samme grunn som de andre lette edelgassene: den danner ikke bindinger som kan brukes til å definere en Pauling-verdi.2. I en gruppe går trenden **motsatt vei**: elektronegativiteten minker nedover. Valenselektronene havner i skall lenger fra kjerna og skjermes av flere fylte innerskall.```pythongruppe17 = grunnstoffer[grunnstoffer["group_id"] == 17]gruppe1  = grunnstoffer[grunnstoffer["group_id"] == 1]plt.plot(gruppe17["atomic_number"], gruppe17["en_pauling"],         marker="o", label="Gruppe 17")plt.plot(gruppe1["atomic_number"], gruppe1["en_pauling"],         marker="s", label="Gruppe 1")plt.xlabel("Atomnummer")plt.ylabel("Elektronegativitet (Pauling)")plt.legend()plt.grid(alpha=0.3)plt.show()```3. Kurven for periode 3 ligger systematisk lavere enn for periode 2, men har samme form. Formen skyldes økende effektiv kjerneladning innafor perioden, mens nivåforskjellen skyldes det ekstra skallet.````

`mendeleev` inneholder langt mer enn elektronegativitet: ioniseringsenergier, ioneradier, isotoper, oksidasjonstall og en del til. Du får en oversikt over alt som er registrert for ett grunnstoff ved å skrive objektet i en egen celle.

In [ ]:
# Ioniseringsenergier er en dictionary med ioniseringsgrad som nøkkelnatrium = element("Na")print("1. ioniseringsenergi:", natrium.ionenergies[1], "eV")print("2. ioniseringsenergi:", natrium.ionenergies[2], "eV")# Skriv objektet alene i en celle for å se alt som finnes:# natrium

```{admonition} Underveisoppgave: Det store spranget:class: tipHent ut de fem første ioniseringsenergiene til magnesium og plott dem mot ioniseringsgrad.Hvor kommer det store spranget, og hvorfor akkurat der? Hva forteller dette deg om elektronstrukturen til magnesium, og hvorfor magnesium danner Mg²⁺ og ikke Mg³⁺?```````{admonition} Løsningsforslag:class: dropdown```pythonmagnesium = element("Mg")grader = [1, 2, 3, 4, 5]energier = [magnesium.ionenergies[n] for n in grader]plt.plot(grader, energier, marker="o")plt.xlabel("Ioniseringsgrad")plt.ylabel("Ioniseringsenergi (eV)")plt.title("Ioniseringsenergier for magnesium")plt.grid(alpha=0.3)plt.show()```Spranget kommer mellom den andre og den tredje ioniseringa. De to første elektronene tas fra 3s-orbitalen i valensskallet. Det tredje må rives ut av det fylte 2p-skallet, som ligger mye nærmere kjerna og er mye hardere bundet.Nettopp derfor stopper magnesium på Mg²⁺: energien som skal til for å ta det tredje elektronet, blir aldri betalt tilbake av gitterenergi eller hydratiseringsenergi i vanlig kjemi.````

### PubChemPySamme argument, men for forbindelser i stedet for grunnstoff. PubChem er verdens største åpne database over kjemiske forbindelser, med over hundre millioner oppføringer. `pubchempy` er et tynt lag med Python-kode som snakker med PubChem sitt API og gir deg svaret som Python-objekter.Legg merke til forskjellen fra `mendeleev`: her ligger dataene ikke på maskina di, men hentes over nettet mens koden kjører. Det betyr at du trenger internettforbindelse, at det tar litt tid, og at du bør være grei og ikke sende tusenvis av forespørsler i en løkke.

In [ ]:
import pubchempy as pcptreff = pcp.get_compounds("paracetamol", "name")paracetamol = treff[0]print("PubChem CID: ", paracetamol.cid)print("Molekylformel:", paracetamol.molecular_formula)print("Molar masse:  ", paracetamol.molecular_weight, "g/mol")print("IUPAC-navn:   ", paracetamol.iupac_name)

```{admonition} Når API-et endrer seg:class: warningPubChem endrer av og til navn på felt i API-et sitt, og da kan attributter som fungerte i fjor slutte å virke. Hvis en attributt gir feilmelding, sjekk dokumentasjonen til `pubchempy` og PubChem sitt eget API i stedet for å anta at koden din er feil.Dette er ikke et argument mot å bruke biblioteket. Det er en påminnelse om at kode som henter data over nett har en holdbarhetsdato, og at du må kunne lese dokumentasjon for å vedlikeholde den.```

In [ ]:
# CID-en er nøkkelen videre. Vi bruker den igjen i molekylvisualisering.for navn in ["koffein", "aspirin", "ibuprofen", "askorbinsyre"]:    forbindelse = pcp.get_compounds(navn, "name")[0]    print(f"{navn:15} CID {forbindelse.cid:>8}   "          f"{forbindelse.molecular_formula:>10}   "          f"{forbindelse.molecular_weight} g/mol")

```{admonition} Underveisoppgave: Bygg din egen tabell:class: tipLag ei liste med fem legemidler eller naturstoffer du er nysgjerrig på. Hent molekylformel, molar masse og CID for hvert av dem, og samle det i en pandas-dataframe.Sorter tabellen etter molar masse og skriv den ut. Ta vare på CID-ene: du får bruk for dem når vi skal visualisere molekylene.```````{admonition} Løsningsforslag:class: dropdown```pythonimport pandas as pdnavn_liste = ["koffein", "nikotin", "morfin", "penicillin G", "kolesterol"]rader = []for navn in navn_liste:    forbindelse = pcp.get_compounds(navn, "name")[0]    rader.append({        "navn": navn,        "cid": forbindelse.cid,        "formel": forbindelse.molecular_formula,        "molar_masse": float(forbindelse.molecular_weight),    })tabell = pd.DataFrame(rader).sort_values("molar_masse")tabell```````

## Del 2: Bibliotek som gir deg struktur og representasjon### RDKit`RDKit` er standardverktøyet i kjeminformatikk, og det er det biblioteket i dette kapitlet som gir deg desidert mest du ikke kunne skrevet selv.Grunnen er at RDKit forstår **molekylstruktur**, ikke bare formler. Det kan lese en tekststreng, bygge en graf av atomer og bindinger, finne ut hvilke ringer som er aromatiske, tegne en strukturformel med fornuftig plassering av atomene, lete etter funksjonelle grupper, og generere en 3D-konformasjon. Hver enkelt av disse tingene er et lite forskningsfelt.For oss er RDKit også interessant av en annen grunn. Det lar deg bevege deg mellom **representasjonsnivåene** i kjemi med noen få linjer kode:- **Symbolsk:** SMILES-koden `CCO` er etanol.- **Symbolsk, todimensjonal:** strukturformelen som RDKit tegner.- **Submikroskopisk:** en 3D-konformasjon med bindingslengder og vinkler.- **Makroskopisk:** beregna egenskaper som molar masse, logP og polart overflateareal, som kan sammenliknes med målte verdier.Det er den samme forbindelsen hele veien. Det er bare representasjonen som endrer seg.#### SMILESSMILES (Simplified Molecular Input Line Entry System) er en måte å skrive et molekyl som én linje tekst. Reglene i korte trekk:- Atomer skrives med grunnstoffsymbol. Hydrogen skrives vanligvis ikke, det legges til automatisk.- Nabotegn betyr binding: `CCO` er C bundet til C bundet til O, altså etanol.- `=` er dobbeltbinding, `#` er trippelbinding.- Parenteser er sidegrupper: `CC(C)C` er 2-metylpropan.- Tall lukker ringer: `C1CCCCC1` er sykloheksan.- Små bokstaver betyr aromatisk: `c1ccccc1` er benzen.

In [ ]:
from rdkit import Chemfrom rdkit.Chem import Draw, Descriptors, AllChemfrom rdkit.Chem.Draw import IPythonConsole   # gjør at molekyler tegnes automatiskkoffein = Chem.MolFromSmiles("CN1C=NC2=C1C(=O)N(C)C(=O)N2C")koffein

```{admonition} Sjekk alltid at innlesinga gikk bra:class: warning`Chem.MolFromSmiles` returnerer `None` hvis SMILES-koden ikke er gyldig. Den kaster ikke feilmelding. Hvis du sender `None` videre til en annen funksjon, får du en forvirrende feil et helt annet sted i programmet.Ta derfor for vane å teste:```pythonmolekyl = Chem.MolFromSmiles(smiles)if molekyl is None:    print("Ugyldig SMILES:", smiles)``````

#### Flere molekyler samtidig

In [ ]:
smiles = {    "koffein":            "CN1C=NC2=C1C(=O)N(C)C(=O)N2C",    "paracetamol":        "CC(=O)Nc1ccc(O)cc1",    "acetylsalisylsyre":  "CC(=O)Oc1ccccc1C(=O)O",    "ibuprofen":          "CC(C)Cc1ccc(cc1)C(C)C(=O)O",}molekyler = [Chem.MolFromSmiles(s) for s in smiles.values()]Draw.MolsToGridImage(molekyler, legends=list(smiles.keys()),                     molsPerRow=4, subImgSize=(240, 200))

#### Fra struktur til egenskapNår RDKit først har strukturen, kan den regne ut en lang rekke deskriptorer. Noen av dem er rene opptellinger, andre er empiriske modeller som er tilpassa eksperimentelle data.

In [ ]:
import pandas as pdrader = []for navn, s in smiles.items():    molekyl = Chem.MolFromSmiles(s)    rader.append({        "navn": navn,        "molar_masse": round(Descriptors.MolWt(molekyl), 2),        "logP": round(Descriptors.MolLogP(molekyl), 2),        "TPSA": round(Descriptors.TPSA(molekyl), 1),        "H-donorer": Descriptors.NumHDonors(molekyl),        "H-akseptorer": Descriptors.NumHAcceptors(molekyl),    })pd.DataFrame(rader)

Legg merke til hva de ulike kolonnene faktisk er:- **Molar masse** er en ren opptelling av atommasser. Den er eksakt, gitt strukturen.- **H-donorer** og **H-akseptorer** er opptellinger etter en bestemt definisjon. De er eksakte gitt definisjonen, men definisjonen er et valg.- **TPSA** (topologisk polart overflateareal) er en sum av bidrag per polart atom, kalibrert mot beregna overflater.- **logP** er en *modell*. Den estimerer fordelinga mellom oktanol og vann ut fra hvilke atomgrupper molekylet inneholder. Den er tilpassa måledata og kan bomme, særlig på uvanlige strukturer.Dette er en viktig sondring. To av kolonnene er fakta om strukturen, to er modellresultater. Et bibliotek gir deg begge deler i samme tabell uten å si fra om forskjellen. Det må du vite selv.

```{admonition} Underveisoppgave: Lipinski:class: tipLipinskis "regel om fem" er en tommelfingerregel for om en forbindelse har egenskaper som ligner på et legemiddel som kan tas som tablett. En forbindelse bryter regelen hvis mer enn ett av følgende er sant:- molar masse over 500 g/mol- logP over 5- flere enn 5 H-bindingsdonorer- flere enn 10 H-bindingsakseptorer1. Skriv en funksjon `bryter_lipinski(smiles)` som returnerer antall brudd.2. Test den på de fire forbindelsene ovenfor.3. Test den så på kolesterol (`CC(C)CCCC(C)C1CCC2C1(CCC3C2CC=C4C3(CCC(C4)O)C)C`) og på et lite peptid du finner SMILES for. Hva ser du?4. Regelen kalles en *tommelfingerregel*. Finn minst ett kjent legemiddel som bryter den. Hva sier det om hvor mye vekt du bør legge på slike regler?```````{admonition} Løsningsforslag:class: dropdown```pythondef bryter_lipinski(smiles):    molekyl = Chem.MolFromSmiles(smiles)    if molekyl is None:        raise ValueError("Ugyldig SMILES: " + smiles)    brudd = 0    if Descriptors.MolWt(molekyl) > 500:        brudd += 1    if Descriptors.MolLogP(molekyl) > 5:        brudd += 1    if Descriptors.NumHDonors(molekyl) > 5:     brudd += 1    if Descriptors.NumHAcceptors(molekyl) > 10: brudd += 1    return bruddfor navn, s in smiles.items():    print(f"{navn:20} {bryter_lipinski(s)} brudd")```De fire småmolekylene bryter ingen kriterier. Kolesterol bryter logP-kriteriet: det er svært upolart. Et peptid vil typisk bryte flere kriterier samtidig.Regelen har mange unntak. Antibiotika, kreftmedisiner og alle legemidler som gis som injeksjon i stedet for tablett, bryter den rutinemessig. En tommelfingerregel er et *filter for å prioritere*, ikke en naturlov.````

#### Å lete etter funksjonelle grupperHer får du noe et bibliotek er svært godt egna til: å søke etter et strukturmønster i et molekyl. Mønsteret skrives i SMARTS, som er SMILES utvida med jokertegn og betingelser.

In [ ]:
monstre = {    "karboksylsyre":  "[CX3](=O)[OX2H1]",    "ester":          "[CX3](=O)[OX2][CX4]",    "amid":           "[CX3](=O)[NX3]",    "alkohol/fenol":  "[OX2H]",    "aromatisk ring": "c1ccccc1",}rader = []for navn, s in smiles.items():    molekyl = Chem.MolFromSmiles(s)    rad = {"navn": navn}    for gruppe, smarts in monstre.items():        mal = Chem.MolFromSmarts(smarts)        rad[gruppe] = len(molekyl.GetSubstructMatches(mal))    rader.append(rad)pd.DataFrame(rader)

```{admonition} Underveisoppgave: Kontroller maskina:class: tipTabellen ovenfor er generert av et program. Kontroller den med kjemikunnskapen din.1. Tegn acetylsalisylsyre for hånd og tell etter. Stemmer antallet ester- og karboksylsyregrupper?2. Paracetamol har ifølge tabellen én amidgruppe. Ser du den i strukturformelen?3. `[OX2H]`-mønsteret teller også OH-gruppa i en karboksylsyre. Er det riktig eller feil? Diskuter hvorfor et program ikke kan svare på det spørsmålet uten at du forteller det hva du er ute etter.4. Legg til et mønster for **keton** og ett for **eter**. Test på de fire forbindelsene. Gikk det som du trodde?```````{admonition} Løsningsforslag:class: dropdown1. Ja. Acetylsalisylsyre har én ester (acetylgruppa på fenol-oksygenet) og én karboksylsyre. Dette er en fin sjekk på at både du og programmet har rett.2. Ja, `CC(=O)N`-delen. Paracetamol er et acetamid.3. Begge deler, avhengig av hva du spør om. `[OX2H]` betyr "oksygen med to bindinger, hvorav ett hydrogen", og det er sant for karboksylsyrens OH. Vil du ha *bare* alkoholer, må du utelukke karbonylnaboen eksplisitt, for eksempel med `[OX2H][CX4]`. Programmet gjør akkurat det du ber om, ikke det du mener.4. Keton er vanskeligere enn det ser ut: `[#6][CX3](=O)[#6]` treffer også karbonylet i en ester eller et amid hvis du ikke er presis nok. Dette er hovedgrunnen til at SMARTS-mønstre bør testes mot molekyler der du kjenner fasiten.````

#### Fra 2D til 3DEn SMILES-kode sier ingenting om geometri. RDKit kan generere en rimelig 3D-struktur ved først å legge til hydrogenatomene, så plassere atomene med en avstandsgeometrimetode, og til slutt optimere geometrien med et kraftfelt.Resultatet er en **konformasjon**, ikke fasiten. Det er én rimelig geometri, funnet med en klassisk modell uten kvantemekanikk.

In [ ]:
etanol = Chem.AddHs(Chem.MolFromSmiles("CCO"))AllChem.EmbedMolecule(etanol, randomSeed=42)AllChem.MMFFOptimizeMolecule(etanol)molblokk = Chem.MolToMolBlock(etanol)print(molblokk[:300])

Denne `molblokk`-strengen er et standardformat som andre program forstår. Vi bruker den i neste kapittel til å tegne molekylet i 3D. Her har du et konkret eksempel på det tredje argumentet for å bruke bibliotek: **standardisering**. RDKit og visualiseringsbibliotekene har aldri sett hverandre, men de er enige om formatet.

## Del 3: Beregninger du bør gjøre selvNå snur vi argumentet. Det finnes flere Python-bibliotek som tilbyr "kjemiberegninger": stoffmengde, fortynning, balansering av likninger, cellepotensialer. De ser praktiske ut, og det er en fristelse å ta dem i bruk.Vi lar være, og det er to grunner til det.**Den første er faglig.** `stoffmengde(masse=2, formel="C4H10O")` er `n = m / M`. Det er én linje kode og det er kjernen i støkiometri. Pakker du den inn i et bibliotek, bytter du bort forståelse mot et API du aldri får bruk for igjen. Dette er nøyaktig samme argument som i kapittel 1: vi lager algoritmene selv fordi vi forstår dem bedre da, og fordi vi da kan endre dem når vi trenger noe litt annet.**Den andre er praktisk.** Små kjemibibliotek har en tendens til å bli forlatt. Et bibliotek som ikke har fått en ny versjon på flere år, ryker før eller siden mot nyere versjoner av `numpy` eller `pandas`. Da sitter du med et emne der halvparten av kodeeksemplene ikke kjører.En tredje ting er verdt å nevne, og er lettere å overse. Noen av disse bibliotekene skriver ut kjemiske formler på former som `1H₁I₁` og `Na₁N₁O₃`. Subskript 1 er ikke konvensjonell kjemisk notasjon. Et verktøy som modellerer feil notasjon for deg, jobber mot deg.### Støkiometri du skriver selvMolar masse er det eneste vi trenger data til, og den har vi allerede fra RDKit eller mendeleev. Resten er kjemi.

In [ ]:
from rdkit import Chemfrom rdkit.Chem import Descriptorsdef molar_masse(smiles):    """Molar masse i g/mol for et molekyl gitt ved SMILES."""    molekyl = Chem.MolFromSmiles(smiles)    if molekyl is None:        raise ValueError("Ugyldig SMILES: " + smiles)    return Descriptors.MolWt(molekyl)def stoffmengde(masse, smiles):    """Stoffmengde i mol for en gitt masse i gram."""    return masse / molar_masse(smiles)def masse(stoffmengde_mol, smiles):    """Masse i gram for en gitt stoffmengde i mol."""    return stoffmengde_mol * molar_masse(smiles)butan_1_ol = "CCCCO"print("Molar masse:", round(molar_masse(butan_1_ol), 2), "g/mol")print("2.00 g tilsvarer", round(stoffmengde(2.00, butan_1_ol), 5), "mol")print("0.150 mol veier", round(masse(0.150, butan_1_ol), 3), "g")

```{admonition} Underveisoppgave: Bygg ut verktøykassa di:class: tipSkriv dine egne funksjoner for følgende, med docstring og fornuftige parameternavn:1. `konsentrasjon(masse, smiles, volum_liter)` som gir molaritet.2. `fortynn(c1, v1, v2)` som gir sluttkonsentrasjonen etter fortynning.3. `antall_molekyler(masse, smiles)` som bruker Avogadros tall.4. `masseprosent(smiles, grunnstoffsymbol)` som gir masseprosenten av ett grunnstoff i forbindelsen. Til denne trenger du å iterere over atomene i molekylet med `molekyl.GetAtoms()` og `atom.GetSymbol()`, og du må huske på hydrogenatomene som ikke er skrevet ut. Se på `Chem.AddHs`.Test hver funksjon mot en beregning du gjør for hånd. Det er hele poenget med å skrive dem selv.```````{admonition} Løsningsforslag:class: dropdown```pythonAVOGADRO = 6.02214076e23   # per mol, eksakt definertdef konsentrasjon(masse, smiles, volum_liter):    """Molaritet i mol/L."""    return stoffmengde(masse, smiles) / volum_literdef fortynn(c1, v1, v2):    """Sluttkonsentrasjon etter fortynning fra volum v1 til v2."""    return c1 * v1 / v2def antall_molekyler(masse, smiles):    """Antall molekyler i en gitt masse."""    return stoffmengde(masse, smiles) * AVOGADROdef masseprosent(smiles, grunnstoffsymbol):    """Masseprosent av ett grunnstoff i en forbindelse."""    from mendeleev import element    molekyl = Chem.AddHs(Chem.MolFromSmiles(smiles))    total = 0.0    valgt = 0.0    for atom in molekyl.GetAtoms():        symbol = atom.GetSymbol()        m = element(symbol).atomic_weight        total += m        if symbol == grunnstoffsymbol:            valgt += m    return 100 * valgt / totalprint(round(masseprosent("CCCCO", "C"), 2), "% karbon i butan-1-ol")```Kontroll for hånd: butan-1-ol er C4H10O med M = 74.12 g/mol. Karbon bidrar med 4 · 12.011 = 48.04 g/mol, altså 64.8 %.````

### Balansering som lineær algebraÅ balansere en reaksjonslikning er det eneste i denne kategorien som er *litt* vanskelig. Men det er ikke vanskelig fordi det krever mye kode. Det er vanskelig fordi det krever at du ser hva problemet egentlig er.En balansert likning er en påstand om at hvert grunnstoff er bevart. Med ukjente koeffisienter blir det én likning per grunnstoff. Det er altså et **homogent lineært likningssystem**, og løsninga er nullrommet til koeffisientmatrisa.La oss balansere ufullstendig forbrenning av benzen, der vi får CO og vann:$$a \cdot \mathrm{C_6H_6} + b \cdot \mathrm{O_2} \longrightarrow c \cdot \mathrm{CO} + d \cdot \mathrm{H_2O}$$Én likning per grunnstoff, med produktene på venstre side og negativt fortegn:- Karbon: $6a - c = 0$- Hydrogen: $6a - 2d = 0$- Oksygen: $2b - c - d = 0$

In [ ]:
import sympy as sp#                a   b   c   dA = sp.Matrix([[ 6,  0, -1,  0],    # C               [ 6,  0,  0, -2],    # H               [ 0,  2, -1, -1]])   # Olosning = A.nullspace()[0]# Nullrommet gir en retning, ikke bestemte tall. Vi skalerer opp til# minste sett med hele tall ved å gange med fellesnevneren.nevnere = [sp.Rational(x).q for x in losning]koeffisienter = losning * sp.ilcm(*nevnere)print("a, b, c, d =", list(koeffisienter))

Svaret er $2\,\mathrm{C_6H_6} + 9\,\mathrm{O_2} \longrightarrow 12\,\mathrm{CO} + 6\,\mathrm{H_2O}$.Kontroller det: 12 karbon på hver side, 12 hydrogen på hver side, 18 oksygen på hver side.Legg merke til hva du faktisk gjorde her. Den eneste vanskelige delen var å sette opp matrisa, og det er ren kjemi. `sympy` gjorde bare den lineære algebraen. Det er en helt annen arbeidsdeling enn å kalle `reaksjon.balance()` og håpe på det beste.

```{admonition} Underveisoppgave: Balanser tre til:class: tipSett opp matrisa og balanser følgende med metoden ovenfor. Kontroller hvert svar for hånd.1. Fullstendig forbrenning av etanol: $\mathrm{C_2H_5OH} + \mathrm{O_2} \rightarrow \mathrm{CO_2} + \mathrm{H_2O}$2. Framstilling av ammoniakk: $\mathrm{N_2} + \mathrm{H_2} \rightarrow \mathrm{NH_3}$3. En redoksreaksjon i sur løsning: $\mathrm{MnO_4^-} + \mathrm{Fe^{2+}} + \mathrm{H^+} \rightarrow \mathrm{Mn^{2+}} + \mathrm{Fe^{3+}} + \mathrm{H_2O}$Den siste krever noe mer: du må ta med **ladning** som en ekstra rad i matrisa, på samme måte som et grunnstoff. Hvorfor fungerer det?```````{admonition} Løsningsforslag:class: dropdownEtanol, med rekkefølgen (C2H5OH, O2, CO2, H2O):```pythonA = sp.Matrix([[ 2,  0, -1,  0],    # C               [ 6,  0,  0, -2],    # H               [ 1,  2, -2, -1]])   # O```Dette gir 1, 3, 2, 3.For redoksreaksjonen med rekkefølgen (MnO4-, Fe2+, H+, Mn2+, Fe3+, H2O):```pythonA = sp.Matrix([[ 1,  0,  0, -1,  0,  0],   # Mn               [ 4,  0,  0,  0,  0, -1],   # O               [ 0,  1,  0,  0, -1,  0],   # Fe               [ 0,  0,  1,  0,  0, -2],   # H               [-1,  2,  1, -2, -3,  0]])  # ladning```Svaret er 1, 5, 8, 1, 5, 4.Ladning fungerer som en ekstra rad fordi ladningsbevaring er en bevaringslov av nøyaktig samme *matematiske* form som massebevaring: summen av ladning på venstre side må være lik summen på høyre. Matrisa bryr seg ikke om hva rada betyr fysisk, bare at det er en størrelse som skal balansere.````

## Del 4: Numerikk du bør sette utSå var det den siste kategorien: beregninger som er så tunge at det ville vært dumt å skrive dem selv hver gang, men der du likevel bør vite hva som skjer.### pHcalcÅ regne ut pH i en løsning av en sterk syre er trivielt. Å regne ut pH i en løsning av en toprotisk syre med en buffer og et inert salt er det ikke. Da må du løse **ladningsbalansen** for hele systemet samtidig, og det blir fort en likning som ikke lar seg løse med algebra.`pHcalc` gjør akkurat dette. Det definerer tre klasser:- `Acid` for en syre med én eller flere Ka-verdier- `Inert` for et ion som ikke deltar i protolyse, som Na⁺ eller Cl⁻- `System` for løsninga som helhetH₃O⁺ og OH⁻ defineres aldri eksplisitt. Konsentrasjonen av H₃O⁺ er den ukjente som justeres til ladningsbalansen går opp, og OH⁻ følger av vannets ionprodukt.

In [ ]:
from pHcalc import Acid, Inert, System# 0.010 M eddiksyre. Ka = 1.75e-5, og HA har ladning 0.eddiksyre = Acid(Ka=1.75e-5, charge=0, conc=0.010)losning = System(eddiksyre)losning.pHsolve()print("pH =", round(losning.pH, 3))

Her kommer koblinga til kapitlet om numeriske metoder. `pHcalc` løser dette med en numerisk nullpunktsalgoritme, av samme familie som halveringsmetoden og Newtons metode du har implementert selv. Vi kan gjøre nøyaktig det samme for hånd og se om vi får samme svar.For en enprotisk syre er ladningsbalansen:$$[\mathrm{H_3O^+}] = [\mathrm{A^-}] + [\mathrm{OH^-}]$$Med $[\mathrm{A^-}] = \frac{K_a \cdot c}{K_a + [\mathrm{H_3O^+}]}$ og $[\mathrm{OH^-}] = K_w / [\mathrm{H_3O^+}]$ blir det et nullpunktsproblem i $[\mathrm{H_3O^+}]$.

In [ ]:
import numpy as npdef ladningsbalanse(h, Ka, c, Kw=1.0e-14):    """Skal være null når h er riktig H3O+-konsentrasjon."""    A_minus = Ka * c / (Ka + h)    OH = Kw / h    return h - A_minus - OHdef halveringsmetoden(f, a, b, tol=1e-18, n=200):    """Finner nullpunktet til f i intervallet [a, b]."""    for _ in range(n):        c = (a + b) / 2        if abs(f(c)) < tol:            return c        if f(a) * f(c) < 0:            b = c        else:            a = c    return (a + b) / 2h = halveringsmetoden(lambda x: ladningsbalanse(x, 1.75e-5, 0.010),                      1e-14, 1.0)print("pH med egen kode:", round(-np.log10(h), 3))print("pH med pHcalc:   ", round(losning.pH, 3))

Dette er den beste grunnen til å bruke et bibliotek: du har sett hva som er inni boksen, og du kan kontrollere at boksen gjør det den skal.### Sammensatte systemerNå er det verdt å ta i bruk biblioteket, for her blir det tungt å skrive selv. Fosforsyre er treprotisk, og systemet får flere koblede likevekter samtidig.

In [ ]:
# Fosforsyre, treprotisk. Ka-verdiene gis som ei liste.fosforsyre = Acid(Ka=[7.5e-3, 6.2e-8, 4.8e-13], charge=0, conc=0.010)losning = System(fosforsyre)losning.pHsolve()print("0.010 M H3PO4:  pH =", round(losning.pH, 3))# Natriumdihydrogenfosfat: samme syre, men delvis nøytralisert.# Ett Na+ per formelenhet.natrium = Inert(charge=1, conc=0.010)losning2 = System(fosforsyre, natrium)losning2.pHsolve()print("0.010 M NaH2PO4: pH =", round(losning2.pH, 3))

### TitrerkurverEn titrerkurve er bare pH beregna for mange ulike mengder tilsatt base. Med `pHcalc` blir det ei løkke.

In [ ]:
import matplotlib.pyplot as pltc_syre = 0.010na_konsentrasjoner = np.linspace(1e-8, 0.020, 300)pH_verdier = []for c_na in na_konsentrasjoner:    syre = Acid(Ka=1.75e-5, charge=0, conc=c_syre)    base = Inert(charge=1, conc=c_na)    system = System(syre, base)    system.pHsolve()    pH_verdier.append(system.pH)plt.figure(figsize=(7, 4))plt.plot(na_konsentrasjoner / c_syre, pH_verdier, color="teal")plt.axvline(1.0, color="grey", linestyle="--", label="Ekvivalenspunkt")plt.xlabel("Mol NaOH per mol syre")plt.ylabel("pH")plt.title("Titrering av 0.010 M eddiksyre med NaOH")plt.legend()plt.grid(alpha=0.3)plt.show()

```{admonition} Underveisoppgave: Les kurven:class: tip1. Hvor på kurven ligger bufferområdet? Hva er pH i midten av det, og hva er sammenhengen med pKa til eddiksyre?2. Hvorfor er pH ved ekvivalenspunktet *over* 7, og ikke lik 7?3. Lag samme kurve for saltsyre (bruk `Inert(charge=-1, conc=0.010)` for Cl⁻). Hvordan skiller den seg fra kurven for eddiksyre, og hvorfor?4. Lag kurven for fosforsyre. Hvor mange sprang ser du, og hvorfor ser du ikke like mange sprang som syren har protoner?```````{admonition} Løsningsforslag:class: dropdown1. Bufferområdet er det flate partiet rundt halv nøytralisering, altså rundt 0.5 på x-aksen. Der er pH lik pKa, som for eddiksyre er omtrent 4.76.2. Ved ekvivalenspunktet er all syra omdanna til acetat, som er den korresponderende basen til en svak syre. Acetat reagerer med vann og gir OH⁻, så løsninga blir basisk.3. For saltsyre er det ingen buffersone. Kurven stiger nesten rett gjennom hele området og ekvivalenspunktet ligger på pH 7, fordi Cl⁻ ikke er en base av betydning.4. Fosforsyre gir to tydelige sprang, ikke tre. Det tredje pKa-trinnet ligger på omtrent 12.3, og der er løsninga så basisk at vannets egen protolyse dominerer og jevner ut spranget.````

## Å vurdere et bibliotekDet viktigste du tar med deg fra dette kapitlet er ikke syntaksen til fire bibliotek. Det er en vane: å stille noen få spørsmål før du tar et nytt bibliotek i bruk.**1. Når kom siste versjon?** Gå til prosjektet på PyPI eller GitHub og se på datoen. Et kjemibibliotek som ikke har fått en oppdatering på tre til fire år er sannsynligvis forlatt, og vil før eller siden ryke mot nyere `numpy` eller `pandas`.**2. Hva gir det meg som jeg ikke kan skrive selv?** Data, tung maskineri eller et standardformat er gode svar. "Det gjør `n = m/M` for meg" er ikke det.**3. Hvor mange avhengigheter har det?** Et bibliotek som bare bruker `numpy` og `scipy` ryker sjelden. Et som drar inn ti pakker, ryker ofte.**4. Kan jeg kontrollere svaret?** Dette er det viktigste. Finn et tilfelle der du kjenner fasiten, enten fra en håndregning eller fra tabellverdier, og sjekk biblioteket mot den før du stoler på det med noe du ikke kan kontrollere.```{admonition} Dette gjelder også KI-generert kode:class: importantEn språkmodell skriver gjerne kode som bruker et bibliotek du ikke har hørt om, med et API som ser helt plausibelt ut. Noen ganger finnes ikke funksjonen den bruker. Andre ganger finnes den, men gjør noe litt annet enn den ser ut til.De fire spørsmålene ovenfor fungerer like godt på KI-generert kode som på kode du finner på nett. Og punkt 4 er fortsatt det viktigste: kontroller svaret mot noe du kjenner.```

```{admonition} Underveisoppgave: Vurder et ukjent bibliotek:class: tipSøk opp et Python-bibliotek for kjemi som ikke er nevnt i dette kapitlet. Forslag: `chempy`, `periodictable`, `molmass`, `pymatgen`, `ase` eller `cclib`.Skriv en kort vurdering på fem til ti setninger:1. Når kom siste versjon?2. Hvilken av de fire kategoriene i dette kapitlet hører det hjemme i?3. Hva gir det deg som du ikke kan skrive selv?4. Finn ett eksempel i dokumentasjonen, kjør det, og kontroller svaret mot noe du kan regne ut eller slå opp.5. Ville du brukt det? Begrunn.```

## SluttoppgaverDisse oppgavene kombinerer flere av bibliotekene og krever at du bruker det du har lært om datahåndtering tidligere i emnet.```{admonition} Oppgave 1: Trender i periodesystemet:class: tipBruk `fetch_table` til å hente hele periodesystemet.1. Lag et plott av kovalent radius mot atomnummer for grunnstoff 1 til 86. Fargelegg punktene etter blokk (s, p, d, f).2. Marker starten på hver nye periode med en loddrett strek.3. Beskriv sagtannmønsteret du ser og forklar det kjemisk.4. Lag et andre plott av første ioniseringsenergi mot atomnummer i samme figur, med egen y-akse. Hva er sammenhengen mellom de to kurvene, og hvorfor?5. Finn de tre grunnstoffene som avviker mest fra den generelle trenden i ioniseringsenergi innafor periode 2. Forklar hvert avvik.``````{admonition} Oppgave 2: Fra navn til struktur til egenskap:class: tipVelg ti legemidler eller naturstoffer.1. Hent CID, molekylformel og molar masse fra PubChem.2. Hent SMILES for hver av dem (fra PubChem eller ved å slå det opp) og les dem inn i RDKit.3. Kontroller at molar masse fra PubChem og fra RDKit stemmer overens. Hvis de avviker, finn ut hvorfor. (Hint: se på hva som skjer med salter og hydrater.)4. Bygg en dataframe med minst fem RDKit-deskriptorer.5. Lag et spredningsplott av logP mot TPSA og merk punktene med navn. Ser du noe mønster?6. Tegn alle ti strukturene i et rutenett med `Draw.MolsToGridImage`.``````{admonition} Oppgave 3: Titrering med to metoder:class: tipDu skal titrere 25.0 mL 0.100 M maursyre (Ka = 1.8 · 10⁻⁴) med 0.100 M NaOH.1. Beregn pH før tilsetting, ved halv nøytralisering, ved ekvivalenspunktet og etter 5 mL overskudd, ved hjelp av **din egen** nullpunktsalgoritme fra kapitlet om numeriske metoder.2. Beregn de samme fire punktene med `pHcalc`.3. Lag hele titrerkurven med `pHcalc` og marker de fire punktene på den.4. Sammenlikn de to metodene. Hvor stort er avviket, og hva skyldes det?5. Diskuter kort: i hvilke situasjoner ville du brukt din egen kode, og i hvilke ville du brukt biblioteket?``````{admonition} Oppgave 4: Balansering og utbytte:class: tipTermitt-reaksjonen er $\mathrm{Fe_2O_3} + \mathrm{Al} \rightarrow \mathrm{Fe} + \mathrm{Al_2O_3}$.1. Balanser den med matrisemetoden fra del 3.2. Skriv en funksjon som tar balanserte koeffisienter, molare masser og utgangsmasser, og finner ut hvilket stoff som er begrensende reaktant.3. Beregn teoretisk utbytte av jern når du starter med 50.0 g Fe₂O₃ og 20.0 g Al.4. Utvid funksjonen slik at den også håndterer prosentvis utbytte når du oppgir faktisk utbytte.5. Test funksjonen din på minst to andre reaksjoner der du kjenner fasiten.``````{admonition} Oppgave 5: Din egen bibliotekvurdering:class: tipFinn et kjemibibliotek som du mener burde vært med i dette kapitlet, eller et som du mener burde vært advart mot.Skriv en kort tekst på en halv til én side der du:1. Beskriver hva biblioteket gjør, med minst ett kjørende kodeeksempel.2. Plasserer det i en av de fire kategoriene.3. Vurderer det etter de fire spørsmålene i avsnittet ovenfor.4. Konkluderer med en anbefaling.Ta med kontrollen du gjorde av svaret. En vurdering uten en kontroll er bare en mening.```